# 학교명 사전(gazetteer) 구축 — High_Middle_Elementary.csv + University.csv

- 초중고 데이터 (update : 2026-05-06) : https://www.data.go.kr/data/15021148/standard.do
- 대학교 데이터 (update : 2026-03-19) : https://www.data.go.kr/data/15107737/standard.do?recommendDataYn=Y#layer_data_infomation

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("repo 루트를 못 찾았어요 (data/, src/ 폴더 기준)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.append(str(REPO_ROOT / "src"))

REPO_ROOT

WindowsPath('d:/Study/dongguk_university/dreampath')

In [2]:
import pandas as pd

from preprocessing import strip_brackets

## 1. 초/중/고 — High_Middle_Elementary.csv

In [3]:
notouni_raw = pd.read_csv(
    REPO_ROOT / "data" / "raw" / "High_Middle_Elementary.csv", encoding="utf-8-sig"
)

# 폐교 등 운영 중이 아닌 학교는 제외 (필터링용으로만 쓰고 결과엔 컬럼으로 안 남김)
notouni_raw = notouni_raw[notouni_raw["운영상태"] == "운영"]

notouni_df = (
    notouni_raw["학교명"]
    .apply(strip_brackets)
    .drop_duplicates()
    .reset_index(drop=True)
    .to_frame(name="학교명")
)
notouni_df.shape

(10470, 1)

## 2. 대학교 — University.csv

In [4]:
uni_raw = pd.read_csv(REPO_ROOT / "data" / "raw" / "University.csv", encoding="utf-8-sig")

uni_df = (
    uni_raw["학교명"]
    .apply(strip_brackets)
    .drop_duplicates()
    .reset_index(drop=True)
    .to_frame(name="학교명")
)
uni_df.shape

(1969, 1)

## 3. 합치기 + 저장

In [5]:
school_df = pd.concat([notouni_df, uni_df], ignore_index=True).drop_duplicates().reset_index(drop=True)
school_df.shape

(12439, 1)

In [ ]:
output_path = REPO_ROOT / "data" / "processed" / "gt_schoolnames.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
school_df.to_csv(output_path, index=False, encoding="utf-8-sig")
output_path